In [1]:
import shap
import lightgbm as lgb
import copy
import json
import numpy as np
import random
from extract_data_from_booster import get_tree_estimators_from_booster
from load_data import load_data_with_folds

def create_shap_drop_callback(X_train, y_train, drop_rate, skip_drop, max_drop, learning_rate, use_shap, 
                              shap_values_container=None, drop_seed=None, fold_idx=None):
    
    if shap_values_container is None:
        shap_values_container = [None]
    
    def shap_drop_callback(iteration, userdata):
        booster = userdata
        
        # Get current number of iterations (not total trees)
        num_trees = booster.num_trees()
        
        # Skip drop if no trees yet
        if num_trees <= 0:
            booster.set_dart_drop_indices([])
            return

        current_shap = shap_values_container[0]

        # Create unique seed: combine fold_idx (if available) with iteration
        # This ensures different folds get different random sequences
        if drop_seed is not None:
            seed = drop_seed
        elif fold_idx is not None:
            # Use fold_idx * large_number + iteration to create unique seeds per fold
            seed = fold_idx * 10000 + iteration
        else:
            seed = iteration

        # Determine drop indices
        drop_indices = determine_drop_indices(
            booster=booster,
            num_trees=num_trees,
            drop_rate=drop_rate,
            skip_drop=skip_drop,
            max_drop=max_drop,
            X_train=X_train,
            previous_shap_values=current_shap,
            use_shap=use_shap,
            drop_seed=seed
        )

        # Set the drop indices
        print(drop_indices)
        booster.set_dart_drop_indices(drop_indices)
        
        if drop_indices:
            print(f"DART iteration {iteration}: Dropping trees {drop_indices}")
        else:
            print(f"DART iteration {iteration}: No trees dropped")
    
    return shap_drop_callback

def determine_drop_indices(booster, num_trees, drop_rate, skip_drop, max_drop, X_train,
                          previous_shap_values=None, use_shap=False, drop_seed=None):
    # Check skip_drop probability
    rng = np.random.RandomState(drop_seed)
    if rng.rand() < skip_drop:
        return []
    
    # Calculate effective drop rate
    effective_drop_rate = drop_rate
    if max_drop > 0 and num_trees > 0:
        effective_drop_rate = min(drop_rate, max_drop / num_trees)

    drop_indices = []

    # Future: Use SHAP values to inform drop decisions
    if use_shap and previous_shap_values is not None:
        tree_scores = calculate_tree_scores(booster, previous_shap_values, X_train)
        
        for i in range(num_trees):
            drop_prob = effective_drop_rate * tree_scores[i] * num_trees
            if rng.rand() < drop_prob:
                drop_indices.append(i)
                if max_drop > 0 and len(drop_indices) >= max_drop:
                    break
        
    else:
        # Random selection based on drop rate
        for i in range(num_trees):
            if rng.rand() < effective_drop_rate:
                drop_indices.append(i)

                # Stop if we've reached max_drop
                if max_drop > 0 and len(drop_indices) >= max_drop:
                    break
    
    return drop_indices

def calculate_shap_values(booster, X_train):
    model_str = booster.model_to_string()
    snap_booster = lgb.Booster(model_str=model_str)

    n_trees = snap_booster.num_trees()
    shap_prev = 0
    per_tree = []

    for k in range(1, n_trees + 1):
        shap_k = snap_booster.predict(X_train, num_iteration=k, pred_contrib=True)
        per_tree.append(shap_k - shap_prev)
        shap_prev = shap_k

    del snap_booster

    return np.stack(per_tree, axis=0)

def calculate_tree_scores(booster, shap_values, X_train):
    # Compute mean absolute SHAP values per feature
    total_shap_values = shap_values.sum(axis=0)[:, :-1]
    mean_abs_shap_values = np.mean(np.abs(total_shap_values), axis=0)
    sum_mean_abs_shap_values = np.sum(mean_abs_shap_values)

    prob_shap_values = mean_abs_shap_values / sum_mean_abs_shap_values

    tree_scores = []

    for tree in booster.dump_model()['tree_info']:
        feature_split_count = get_feature_split_count(tree['tree_structure'], X_train)
        tree_score = np.sum(prob_shap_values * feature_split_count)
        tree_scores.append(tree_score)

    tree_scores = np.array(tree_scores)
    sum_scores = np.sum(tree_scores)

    tree_scores = tree_scores / sum_scores

    return tree_scores

def get_feature_split_count(tree_structure, X_train):
    feature_split_count = np.zeros(len(X_train.columns))

    def extract_feature_split_count(tree_structure):
        if 'leaf_index' in tree_structure:
            return
        
        feature_split_count[tree_structure['split_feature']] += 1

        extract_feature_split_count(tree_structure['left_child'])
        extract_feature_split_count(tree_structure['right_child'])

    extract_feature_split_count(tree_structure)

    return feature_split_count
        
        


/Users/larsabbink/Documents/persoonlijk/LightGBM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import lightgbm as lgb
from load_data import get_fold_data
from sklearn.metrics import mean_squared_error

def run_single_fold(X, y, patient_ids, fold_indices, fold_idx, params, num_iterations, use_lightgbm_default, use_shap):
    X_train, y_train, X_test, y_test = get_fold_data(
        X, y, patient_ids, fold_indices, fold_idx
    )

    booster = create_booster_with_callback(
        X_train, y_train, X_test, y_test, params, use_lightgbm_default, use_shap, fold_idx=fold_idx
    )

    train_fold_model(booster, num_iterations, 10, use_shap, X_train)

    results = evaluate_fold(booster, X_train, y_train, X_test, y_test, fold_idx)

    return results

def create_booster_with_callback(X_train, y_train, X_test, y_test, params, use_lightgbm_default, use_shap, fold_idx=None):
    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

    booster = lgb.Booster(params, train_set=train_data)
    booster.add_valid(test_data, "valid")

    shap_values_container = [None]

    if not use_lightgbm_default:
        callback = create_shap_drop_callback(
            X_train=X_train,
            y_train=y_train,
            drop_rate=params.get('drop_rate', 0.1),
            skip_drop=params.get('skip_drop', 0.5),
            max_drop=params.get('max_drop', 0),
            learning_rate=params.get('learning_rate', 0.1),
            use_shap=use_shap,
            shap_values_container=shap_values_container,
            fold_idx=fold_idx
        )
        booster.set_dart_callback(callback, user_data=booster)

    booster._shap_values_container = shap_values_container

    return booster

def train_fold_model(booster, num_iterations, print_interval=10, use_shap = False, X_train = None):
    shap_values_container = getattr(booster, '_shap_values_container', None)

    if use_shap and shap_values_container is None:
        print("Warning: SHAP container not found on booster. SHAP values won't be shared with callback.")
        shap_values_container = [None]
    
    for i in range(num_iterations):
        booster.update()

        if use_shap:
            shap_values = calculate_shap_values(booster, X_train)
            shap_values_container[0] = shap_values

        train_result = booster.eval_train()[0]
        test_result = booster.eval_valid()[0]
        train_mse = float(train_result[2])
        test_mse = float(test_result[2])
        print(f"Iteration {i+1}: Train MSE={train_mse:.4f}, Test MSE={test_mse:.4f}")

def evaluate_fold(booster, X_train, y_train, X_test, y_test, fold_idx):
    y_pred_train = booster.predict(X_train)
    y_pred_test = booster.predict(X_test)
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    return {
        'fold_idx': fold_idx,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_samples': len(X_train),
        'test_samples': len(X_test)
    }

In [3]:
def print_cv_summary(fold_results):
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    for result in fold_results:
        print(f"Fold {result['fold_idx']:2d}: Train MSE={result['train_mse']:.4f}, Test MSE={result['test_mse']:.4f} (Train: {result['train_samples']:5d}, Test: {result['test_samples']:5d})")
    avg_test_mse = np.mean([r['test_mse'] for r in fold_results])
    std_test_mse = np.std([r['test_mse'] for r in fold_results])
    print(f"\nAverage Test MSE: {avg_test_mse:.4f} ± {std_test_mse:.4f}")

In [ ]:
n_folds = 10
random_state = 42

X, y, patient_ids, fold_indices = load_data_with_folds(
    file_path="../data/slice_localization_data.csv",
    target_col="reference",
    drop_cols=["patientId"],
    n_folds=n_folds,
    random_state=random_state
)

params = {
    'objective': 'regression',
    'boosting_type': 'dart',
    'num_leaves': 100,
    'learning_rate': 0.2,
    'feature_fraction_bynode': 0.2,
    'drop_rate': 0.01,
    'metric': 'mse',
    'verbose': 1,
    'max_depth': 10
}

In [5]:
use_lightgbm_default = True
use_shap = False

fold_results = []
for fold_idx in range(n_folds):
    print(f"Running fold {fold_idx + 1} of {n_folds}")
    # Set different drop_seed for each fold to ensure different random sequences
    fold_params = params.copy()
    fold_params['drop_seed'] = 42 + fold_idx  # Different seed per fold
    result = run_single_fold(X, y, patient_ids, fold_indices, fold_idx, fold_params, 25, use_lightgbm_default, use_shap)
    fold_results.append(result)

print_cv_summary(fold_results)

Running fold 1 of 10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032719 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 79665
[LightGBM] [Info] Number of data points in the train set: 47720, number of used features: 378
[LightGBM] [Info] Start training from score 45.972474
[LightGBM] [Info] DART iteration 0: No trees dropped
Iteration 1: Train MSE=330.9716, Test MSE=392.2982
[LightGBM] [Info] DART iteration 1: No trees dropped
Iteration 2: Train MSE=222.8287, Test MSE=290.6773
[LightGBM] [Info] DART iteration 2: No trees dropped
Iteration 3: Train MSE=152.3396, Test MSE=225.5397
[LightGBM] [Info] DART iteration 3: No trees dropped
Iteration 4: Train MSE=104.9420, Test MSE=174.3656
[LightGBM] [Info] DART iteration 4: No trees dropped
Iteration 5: Train MSE=73.8926, Test MSE=140.5477
[LightGBM] [Info] DART iteration 5: No trees droppe

In [6]:
use_lightgbm_default = False
use_shap = True

fold_results = []
for fold_idx in range(n_folds):
    print(f"Running fold {fold_idx + 1} of {n_folds}")
    result = run_single_fold(X, y, patient_ids, fold_indices, fold_idx, params, 25, use_lightgbm_default, use_shap)
    fold_results.append(result)

print_cv_summary(fold_results)

Running fold 1 of 10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031971 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 79665
[LightGBM] [Info] Number of data points in the train set: 47720, number of used features: 378
[LightGBM] [Info] Start training from score 45.972474
[LightGBM] [Info] DART iteration 0: No trees dropped
Iteration 1: Train MSE=330.9716, Test MSE=392.2982
[]
DART iteration 1: No trees dropped
[LightGBM] [Info] DART iteration 1: No trees dropped
Iteration 2: Train MSE=222.8287, Test MSE=290.6773
[]
DART iteration 2: No trees dropped
[LightGBM] [Info] DART iteration 2: No trees dropped
Iteration 3: Train MSE=152.3396, Test MSE=225.5397
[]
DART iteration 3: No trees dropped
[LightGBM] [Info] DART iteration 3: No trees dropped
Iteration 4: Train MSE=104.9420, Test MSE=174.3656
[]
DART iteration 4: No trees dropped
[L

In [7]:
use_lightgbm_default = False
use_shap = False

fold_results = []
for fold_idx in range(n_folds):
    print(f"Running fold {fold_idx + 1} of {n_folds}")
    result = run_single_fold(X, y, patient_ids, fold_indices, fold_idx, params, 25, use_lightgbm_default, use_shap)
    fold_results.append(result)

print_cv_summary(fold_results)

Running fold 1 of 10
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031844 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 79665
[LightGBM] [Info] Number of data points in the train set: 47720, number of used features: 378
[LightGBM] [Info] Start training from score 45.972474
[LightGBM] [Info] DART iteration 0: No trees dropped
Iteration 1: Train MSE=330.9716, Test MSE=392.2982
[]
DART iteration 1: No trees dropped
[LightGBM] [Info] DART iteration 1: No trees dropped
Iteration 2: Train MSE=222.8287, Test MSE=290.6773
[]
DART iteration 2: No trees dropped
[LightGBM] [Info] DART iteration 2: No trees dropped
Iteration 3: Train MSE=152.3396, Test MSE=225.5397
[]
DART iteration 3: No trees dropped
[LightGBM] [Info] DART iteration 3: No trees dropped
Iteration 4: Train MSE=104.9420, Test MSE=174.3656
[]
DART iteration 4: No trees dropped
[L

In [8]:
X_train, y_train, X_test, y_test = get_fold_data(
    X, y, patient_ids, fold_indices, 0
)

In [9]:
def dump_shrinkage(booster):
    tree_info = booster.dump_model()['tree_info']
    shrinkage = []
    for i, tree in enumerate(tree_info):
        shrinkage.append(tree["shrinkage"])

    return shrinkage

def compare_shap_across_iterations(results, rtol=1e-5, atol=1e-8):
    iters = sorted(results.keys())
    comparisons = {}

    for i in range(len(iters) - 1):
        iter_prev = iters[i]
        iter_curr = iters[i + 1]

        sv_prev = results[iter_prev]["shap_values"]
        sv_curr = results[iter_curr]["shap_values"]

        n_trees_prev = sv_prev.shape[0]
        n_trees_curr = sv_curr.shape[0]
        n_common = min(n_trees_prev, n_trees_curr)

        per_tree_equal = []
        for t in range(n_common):
            equal = np.allclose(sv_prev[t], sv_curr[t], rtol=rtol, atol=atol)
            per_tree_equal.append(equal)

        all_equal = all(per_tree_equal)

        comparisons[(iter_prev, iter_curr)] = {
            "per_tree_equal": per_tree_equal,
            "all_equal": all_equal,
        }

    return comparisons

def shap_values_equal(values_1, values_2):
    return np.array_equal(values_1, values_2)


In [10]:
use_lightgbm_default = True
use_shap = False

In [11]:
results = {}

booster = create_booster_with_callback(
    X_train, y_train, X_test, y_test, params, use_lightgbm_default, use_shap
)

# Run booster 5 times since there is no dropout yet.
for i in range(10):
    booster.update()
    shrinkage = dump_shrinkage(booster)
    shap_values = calculate_shap_values(booster, X_train)
    results[i] = {"shap_values": shap_values, "shrinkage": shrinkage}

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031592 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 79665
[LightGBM] [Info] Number of data points in the train set: 47720, number of used features: 378
[LightGBM] [Info] Start training from score 45.972474
[LightGBM] [Info] DART iteration 0: No trees dropped
[LightGBM] [Info] DART iteration 1: No trees dropped
[LightGBM] [Info] DART iteration 2: No trees dropped
[LightGBM] [Info] DART iteration 3: No trees dropped
[LightGBM] [Info] DART iteration 4: No trees dropped
[LightGBM] [Info] DART iteration 5: No trees dropped
[LightGBM] [Info] DART iteration 6: No trees dropped
[LightGBM] [Info] DART iteration 7: No trees dropped
[LightGBM] [Info] DART iteration 8: No trees dropped
[LightGBM] [Info] DART iteration 9: No trees dropped


In [12]:
comparisons = compare_shap_across_iterations(results)

for (i_prev, i_curr), info in comparisons.items():
    print(f"Comparing iteration {i_prev} -> {i_curr}")
    print("  Per-tree equal:", info["per_tree_equal"])
    print("  All common trees equal:", info["all_equal"])

Comparing iteration 0 -> 1
  Per-tree equal: [True]
  All common trees equal: True
Comparing iteration 1 -> 2
  Per-tree equal: [True, True]
  All common trees equal: True
Comparing iteration 2 -> 3
  Per-tree equal: [True, True, True]
  All common trees equal: True
Comparing iteration 3 -> 4
  Per-tree equal: [True, True, True, True]
  All common trees equal: True
Comparing iteration 4 -> 5
  Per-tree equal: [True, True, True, True, True]
  All common trees equal: True
Comparing iteration 5 -> 6
  Per-tree equal: [True, True, True, True, True, True]
  All common trees equal: True
Comparing iteration 6 -> 7
  Per-tree equal: [True, True, True, True, True, True, True]
  All common trees equal: True
Comparing iteration 7 -> 8
  Per-tree equal: [True, True, True, True, True, True, True, True]
  All common trees equal: True
Comparing iteration 8 -> 9
  Per-tree equal: [True, True, True, True, True, True, True, True, True]
  All common trees equal: True


In [13]:
for key in results.keys():
    print(results[key]["shrinkage"])

[1]
[1, 0.2]
[1, 0.2, 0.2]
[1, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
[1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]


In [14]:
second_tree_2 = shap_values_1[1, :]
second_tree_3 = shap_values_2[1, :]

print(np.array_equal(second_tree_2, second_tree_3))


NameError: name 'shap_values_1' is not defined

In [ ]:
booster.update()

[2, 3]
DART iteration 6: Dropping trees [2, 3]
[LightGBM] [Info] DART iteration 6: Drop indices: [2, 3]


False

In [ ]:
for i in range(5):
    print(i)

0
1
2
3
4


In [ ]:
tree_info = booster.dump_model()['tree_info']
for tree in tree_info:
    print(tree["shrinkage"])

1
0.2
0.13333333333333333
0.13333333333333333
0.2
0.2
0.06666666666666667


In [ ]:
# tree_info = booster.dump_model()['tree_info'][0]

# feature_splits = np.zeros(len(X_train.columns))
# tree_structure = tree_info['tree_structure']

# def extract_feature_splits(tree_structure):
#     if 'leaf_index' in tree_structure:
#         return
    
#     feature_splits[tree_structure['split_feature']] += 1

#     extract_feature_splits(tree_structure['left_child'])
#     extract_feature_splits(tree_structure['right_child'])

# extract_feature_splits(tree_structure)

In [ ]:
# total_shap_values = shap_values.sum(axis=0)[:, :-1]
# mean_abs_shap = np.mean(np.abs(total_shap_values), axis=0)
# sum_mean_abs_shap = np.sum(mean_abs_shap)

# prob_shap_values = mean_abs_shap / sum_mean_abs_shap
# tree_score = np.sum(prob_shap_values * feature_splits)



In [ ]:
# from matplotlib import pyplot as plt

In [ ]:
# # mean_abs_shap = np.mean(np.abs(shap_values[0][:, :-1]), axis=0)

# plt.figure(figsize=(10, 6))
# plt.bar(range(len(mean_abs_shap)), mean_abs_shap)
# plt.xlabel('Tree Index')
# plt.ylabel('Mean Absolute SHAP Value')
# plt.title('Mean Absolute SHAP Values for Each Tree')

In [ ]:
# plt.figure(figsize=(10, 6))
# plt.bar(range(len(prob_shap_values)), prob_shap_values)
# plt.xlabel('Tree Index')
# plt.ylabel('Probability of SHAP Value')
# plt.title('Probability of SHAP Values for Each Tree')


Running experiments without dropping trees:

Average Test MSE: 39.2077 ± 25.7408

Running experiments with dropping trees:

Average Test MSE: 33.6378 ± 14.9815

In [ ]:
# First result with xai

# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================
# Fold  0: Train MSE=98.6441, Test MSE=161.2053 (Train: 48645, Test:  4855)
# Fold  1: Train MSE=145.5513, Test MSE=96.1656 (Train: 49017, Test:  4483)
# Fold  2: Train MSE=165.2723, Test MSE=169.7415 (Train: 47515, Test:  5985)
# Fold  3: Train MSE=150.3139, Test MSE=154.4181 (Train: 48424, Test:  5076)
# Fold  4: Train MSE=127.4099, Test MSE=135.4833 (Train: 48726, Test:  4774)
# Fold  5: Train MSE=150.0078, Test MSE=179.3376 (Train: 48349, Test:  5151)
# Fold  6: Train MSE=99.9731, Test MSE=144.8636 (Train: 46700, Test:  6800)
# Fold  7: Train MSE=158.1555, Test MSE=294.0266 (Train: 47875, Test:  5625)
# Fold  8: Train MSE=159.5383, Test MSE=190.0869 (Train: 47475, Test:  6025)
# Fold  9: Train MSE=160.7813, Test MSE=271.3331 (Train: 48774, Test:  4726)

# Average Test MSE: 179.6662 ± 57.2733

# After fixing dubble skip_drop issue:
# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================
# Fold  0: Train MSE=33.6535, Test MSE=59.4321 (Train: 47555, Test:  5945)
# Fold  1: Train MSE=38.2600, Test MSE=105.6944 (Train: 47636, Test:  5864)
# Fold  2: Train MSE=38.6760, Test MSE=76.7686 (Train: 46131, Test:  7369)
# Fold  3: Train MSE=34.2742, Test MSE=22.3494 (Train: 49593, Test:  3907)
# Fold  4: Train MSE=38.2343, Test MSE=64.1733 (Train: 48694, Test:  4806)
# Fold  5: Train MSE=51.1803, Test MSE=128.0605 (Train: 49214, Test:  4286)
# Fold  6: Train MSE=32.1009, Test MSE=79.0412 (Train: 46510, Test:  6990)
# Fold  7: Train MSE=53.8017, Test MSE=128.4302 (Train: 48260, Test:  5240)
# Fold  8: Train MSE=34.3747, Test MSE=39.8293 (Train: 50195, Test:  3305)
# Fold  9: Train MSE=36.5262, Test MSE=63.1424 (Train: 47712, Test:  5788)

# Average Test MSE: 76.6922 ± 33.3353

In [ ]:
# First result without xai

# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================
# Fold  0: Train MSE=113.1773, Test MSE=176.7243 (Train: 48645, Test:  4855)
# Fold  1: Train MSE=118.0341, Test MSE=75.0342 (Train: 49017, Test:  4483)
# Fold  2: Train MSE=116.5843, Test MSE=123.7893 (Train: 47515, Test:  5985)
# Fold  3: Train MSE=116.4755, Test MSE=121.7984 (Train: 48424, Test:  5076)
# Fold  4: Train MSE=116.8167, Test MSE=121.5246 (Train: 48726, Test:  4774)
# Fold  5: Train MSE=115.3226, Test MSE=139.2521 (Train: 48349, Test:  5151)
# Fold  6: Train MSE=114.7665, Test MSE=158.4901 (Train: 46700, Test:  6800)
# Fold  7: Train MSE=114.8483, Test MSE=261.9436 (Train: 47875, Test:  5625)
# Fold  8: Train MSE=114.4022, Test MSE=144.6927 (Train: 47475, Test:  6025)
# Fold  9: Train MSE=114.0462, Test MSE=211.1236 (Train: 48774, Test:  4726)

# Average Test MSE: 153.4373 ± 49.9923
# After fixing dubble skip_drop issue:
